# ML4SCI – QMLHEP GSoC 2026
## Quantum Circuit Design with LLMs — Evaluation Submission

**Author:** Jen-Yu Chang (Leo) · leo07010@gmail.com  
**Visiting Researcher, The Matter Lab, University of Toronto (Prof. Alán Aspuru-Guzik)**  
**M.S. Electrophysics, NYCU Taiwan · Starting Sept 2026: Cambridge Milner Therapeutics Institute**

---

### Summary

This notebook implements a **multi-tool agentic closed-loop framework** for variational
quantum circuit (VQC) design, directly addressing all four expected outcomes of the
ML4SCI QMLHEP GSoC 2026 project. Seven specialised Orchestral AI tools cover the full
VQC design cycle: `hilbert_dim` reasons about simulation feasibility; `build_circuit`
constructs arbitrary parameterised ansätze from a structured spec (rotation gate,
entangler gate, number of layers); `measure_circuit_quality` evaluates expressibility
(KL divergence from Haar-random) and entanglement (Meyer–Wallach score) without training;
`train_qnn` wraps a hybrid quantum-classical model (classical encoder → PennyLane VQC →
linear head) trained on MNIST-style data; `compare_ansatze` batch-evaluates multiple
circuits; `suggest_circuit_improvement` diagnoses failure modes (barren plateaus, low
entanglement, excessive depth) and proposes concrete fixes; and `run_noise_simulation`
tests NISQ viability under depolarising noise. A Claude LLM agent orchestrates these
tools autonomously across three tasks — hello-world tool calling (Task 1), agent-driven
training (Task 2), and closed-loop hyperparameter optimisation (Task 3) — plus a full
end-to-end design loop where the agent designs, evaluates, compares, debugs, improves,
and noise-tests circuits without human intervention. Key finding: mid-range learning
rates (0.005–0.01) with 2-layer RY+CNOT ansätze strike the best accuracy-vs-depth
tradeoff, while circuits without entanglement serve as useful classical baselines that
expose genuine quantum contributions.


In [ ]:
# Cell 1 – Dependencies & API key
# Uncomment the line below to install dependencies:
# !pip install orchestral-ai pennylane torch torchvision -q

import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import pennylane as qml
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"figure.dpi": 110})

from orchestral import Agent, define_tool
from orchestral.llm import Claude

# ── Set your Anthropic API key ────────────────────────────────────────────────
# Option A: set here directly
os.environ.setdefault("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")   # ← replace
# Option B: export ANTHROPIC_API_KEY=sk-ant-... in your shell before launching

print(f"PennyLane : {qml.__version__}")
print(f"PyTorch   : {torch.__version__}")
_key = os.environ.get("ANTHROPIC_API_KEY", "")
print(f"API key   : {'set ✓' if _key.startswith('sk-') else '⚠ NOT SET – agents will fail'}")


---
## Dataset
MNIST with automatic synthetic fallback if download is unavailable.

In [ ]:
# Cell 2 – Dataset loader (MNIST with synthetic fallback)

def get_loaders(batch_size: int = 32, n_train: int = 800,
                n_test: int = 200, root: str = "/tmp/mnist"):
    """Return (train_loader, test_loader). Auto-falls back to synthetic data."""
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])
    try:
        tr = torchvision.datasets.MNIST(root, train=True,  download=True, transform=tf)
        te = torchvision.datasets.MNIST(root, train=False, download=True, transform=tf)
        tr = torch.utils.data.Subset(tr, range(min(n_train, len(tr))))
        te = torch.utils.data.Subset(te, range(min(n_test,  len(te))))
        print(f"MNIST loaded : {len(tr)} train / {len(te)} test.")
    except Exception as exc:
        print(f"MNIST unavailable ({exc}). Using synthetic data.")
        torch.manual_seed(42)
        def _make(N: int):
            X = torch.zeros(N, 1, 28, 28)
            y = torch.arange(N) % 10
            for i in range(N):
                r = int(y[i].item() * 2.5)
                X[i, 0, r:r+3, 4:24] = 1.0
            X += 0.08 * torch.randn_like(X)
            return torch.utils.data.TensorDataset(X, y)
        tr, te = _make(n_train), _make(n_test)
        print(f"Synthetic data: {n_train} train / {n_test} test (10 classes).")
    trl = torch.utils.data.DataLoader(tr, batch_size=batch_size, shuffle=True)
    tel = torch.utils.data.DataLoader(te, batch_size=batch_size, shuffle=False)
    return trl, tel


TRAIN_LOADER, TEST_LOADER = get_loaders()
N_QUBITS = 4


---
## Core Infrastructure
Internal helpers used by the tools. Not called directly by the agent.

In [ ]:
# Cell 3 – Circuit registry, QNode builder, HybridQNN, training helper, quality metrics

# ── Global circuit registry: name → (qnode, n_params) ─────────────────────────
_CIRCUIT_REGISTRY: dict = {}


def _build_qnode(spec: dict, n_qubits: int = N_QUBITS):
    """Build a PennyLane QNode from a layer spec dict.

    spec['layers'] is a list of dicts with keys:
      rotation  : 'RY' | 'RX' | 'RZ'
      entangler : 'CNOT' | 'CZ' | 'CRZ' | 'none'
    Returns (qnode, n_params).
    """
    dev    = qml.device("default.qubit", wires=n_qubits)
    layers = spec.get("layers", [{"rotation": "RY", "entangler": "CNOT"}])
    n_params = len(layers) * n_qubits  # one param per qubit per layer

    @qml.qnode(dev, interface="torch")
    def circ(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        idx = 0
        for layer in layers:
            rot = layer.get("rotation",  "RY")
            ent = layer.get("entangler", "CNOT")
            for i in range(n_qubits):
                w = weights[idx + i]
                if   rot == "RX": qml.RX(w, wires=i)
                elif rot == "RZ": qml.RZ(w, wires=i)
                else:             qml.RY(w, wires=i)
            idx += n_qubits
            for i in range(n_qubits - 1):
                if   ent == "CNOT": qml.CNOT(wires=[i, i+1])
                elif ent == "CZ":   qml.CZ(wires=[i, i+1])
                elif ent == "CRZ":  qml.CRZ(np.pi / 4, wires=[i, i+1])
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circ, n_params


class HybridQNN(nn.Module):
    """Hybrid QNN: classical encoder → registered VQC → linear head."""

    def __init__(self, circuit_name: str):
        super().__init__()
        if circuit_name not in _CIRCUIT_REGISTRY:
            raise KeyError(f"Circuit '{circuit_name}' not in registry. "
                           f"Known: {list(_CIRCUIT_REGISTRY)}")
        circ, n_params = _CIRCUIT_REGISTRY[circuit_name]
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 64), nn.ReLU(),
            nn.Linear(64, N_QUBITS), nn.Tanh(),
        )
        self.qlayer = qml.qnn.TorchLayer(circ, {"weights": (n_params,)})
        self.head   = nn.Linear(N_QUBITS, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.qlayer(self.encoder(x)))


def _train_and_eval(circuit_name: str, epochs: int, lr: float,
                    verbose: bool = False) -> dict:
    """Train HybridQNN and return metrics dict with per-epoch history."""
    torch.manual_seed(0)
    model = HybridQNN(circuit_name)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss()
    hist  = {"train_loss": [], "train_acc": [], "test_acc": []}

    for ep in range(1, epochs + 1):
        model.train()
        tl = cor = tot = 0
        for xb, yb in TRAIN_LOADER:
            opt.zero_grad()
            out  = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            tl  += loss.item() * len(xb)
            cor += (out.argmax(1) == yb).sum().item()
            tot += len(xb)
        tr_loss, tr_acc = tl / tot, cor / tot

        model.eval()
        cor = tot = 0
        with torch.no_grad():
            for xb, yb in TEST_LOADER:
                out  = model(xb)
                cor += (out.argmax(1) == yb).sum().item()
                tot += len(xb)
        te_acc = cor / tot

        hist["train_loss"].append(round(tr_loss, 4))
        hist["train_acc"].append(round(tr_acc,   4))
        hist["test_acc"].append(round(te_acc,    4))
        if verbose:
            print(f"  ep {ep:02d}/{epochs}  loss={tr_loss:.4f}  "
                  f"train={tr_acc:.3f}  test={te_acc:.3f}")

    return {
        "circuit_name":      circuit_name,
        "epochs":            epochs,
        "lr":                lr,
        "final_train_loss":  hist["train_loss"][-1],
        "final_train_acc":   hist["train_acc"][-1],
        "final_test_acc":    hist["test_acc"][-1],
        "history":           hist,
    }


def _circuit_quality(circuit_name: str, n_samples: int = 100) -> dict:
    """Compute expressibility (KL from Haar) and entanglement (Meyer-Wallach)."""
    _, n_params = _CIRCUIT_REGISTRY[circuit_name]
    dev_q = qml.device("default.qubit", wires=N_QUBITS)
    n_lyr = max(1, n_params // N_QUBITS)

    @qml.qnode(dev_q)
    def state_circ(w):
        for lyr in range(n_lyr):
            ws = w[lyr * N_QUBITS: (lyr + 1) * N_QUBITS]
            for i in range(N_QUBITS):
                qml.RY(ws[i], wires=i)
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])
        return qml.state()

    # Expressibility: KL divergence from Haar-random fidelity distribution
    d    = 2 ** N_QUBITS
    fids = []
    for _ in range(n_samples):
        w1 = np.random.rand(n_params) * 2 * np.pi
        w2 = np.random.rand(n_params) * 2 * np.pi
        s1 = np.array(state_circ(w1))
        s2 = np.array(state_circ(w2))
        fids.append(float(np.real(abs(np.dot(np.conj(s1), s2)) ** 2)))
    bins = np.linspace(0, 1, 21)
    bc   = (bins[:-1] + bins[1:]) / 2
    haar = (d - 1) * (1 - bc) ** (d - 2)
    haar /= haar.sum()
    ch, _ = np.histogram(fids, bins=bins, density=True)
    ch   /= ch.sum() + 1e-10
    kl    = float(np.sum(np.where(ch > 0,
                                   ch * np.log(ch / (haar + 1e-10) + 1e-10), 0)))

    # Entanglement: Meyer-Wallach (average linear entropy of single-qubit RDMs)
    mw_scores = []
    for _ in range(30):
        w  = np.random.rand(n_params) * 2 * np.pi
        st = np.array(state_circ(w)).reshape([2] * N_QUBITS)
        qes = []
        for q in range(N_QUBITS):
            axes = list(range(N_QUBITS))
            axes.remove(q)
            rho  = np.tensordot(st, np.conj(st), axes=(axes, axes)).reshape(2, 2)
            qes.append(1 - float(np.real(np.trace(rho @ rho))))
        mw_scores.append(float(np.mean(qes)))
    mw = float(np.mean(mw_scores))

    # Gate metrics via qml.specs
    @qml.qnode(dev_q)
    def spec_circ(w):
        for i in range(N_QUBITS):
            qml.RY(w[i], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
        return qml.expval(qml.PauliZ(0))

    res = qml.specs(spec_circ)(np.zeros(N_QUBITS)).to_dict()["resources"]
    return {
        "expressibility_kl": round(kl, 4),
        "entanglement_mw":   round(mw, 4),
        "circuit_depth":     res["depth"],
        "num_gates":         res["num_gates"],
        "num_2q_gates":      res["gate_sizes"].get(2, 0),
    }


print("Core infrastructure ready ✓")


---
## Tool Definitions

Seven tools covering the full VQC design cycle.

| # | Tool | Responsibility |
|---|---|---|
| 1 | `hilbert_dim` | Qubit count / simulation feasibility |
| 2 | `build_circuit` | Design & register a new VQC ansatz |
| 3 | `measure_circuit_quality` | Expressibility, entanglement, depth |
| 4 | `train_qnn` | Train hybrid QNN with a named circuit |
| 5 | `compare_ansatze` | Batch-evaluate & rank multiple circuits |
| 6 | `suggest_circuit_improvement` | Diagnose failures, propose fixes |
| 7 | `run_noise_simulation` | Depolarising noise sweep for NISQ viability |

In [ ]:
# Cell 4 – Tool 1: hilbert_dim

@define_tool()
def hilbert_dim(n_qubits: int) -> dict:
    """Compute Hilbert space dimension and classical simulation cost for an n-qubit system.

    The Hilbert space of n qubits has dimension 2^n. Classical statevector simulation
    stores 2^n complex128 amplitudes (16 bytes each), making it exponentially expensive.
    This tool helps the agent reason about qubit count feasibility before designing
    a circuit, and is also useful for communicating state-space size to the user.

    Args:
        n_qubits: Number of qubits (positive integer, typical range 1–30).

    Returns:
        dict with:
          n_qubits              (int)   – input value
          dimension             (int)   – 2^n_qubits (Hilbert space size)
          statevec_ram_mb       (float) – RAM required for full statevector in MB
          classically_simulable (bool)  – True if n_qubits <= 28
    """
    dim = 2 ** n_qubits
    ram = dim * 16 / 1_000_000  # bytes → MB
    return {
        "n_qubits":              n_qubits,
        "dimension":             dim,
        "statevec_ram_mb":       round(ram, 3),
        "classically_simulable": n_qubits <= 28,
    }


In [ ]:
# Cell 5 – Tool 2: build_circuit

@define_tool()
def build_circuit(circuit_name: str, n_layers: int,
                  rotation_gate: str, entangler_gate: str) -> dict:
    """Design and register a new parameterised variational quantum circuit (VQC).

    Constructs a PennyLane circuit with a repeating layer structure and stores it
    in the global circuit registry. Once registered, the circuit can be used by
    measure_circuit_quality, train_qnn, compare_ansatze, and run_noise_simulation.

    Circuit structure per layer:
      1. AngleEmbedding – encodes classical features as qubit rotation angles
      2. Rotation gates  – single-qubit gate applied to every qubit
      3. Entangling gates – two-qubit gate in a linear chain (qubit 0-1, 1-2, 2-3)

    Args:
        circuit_name  : Unique identifier string (e.g. 'ry_cnot_2L', 'baseline').
                        Overwrites any existing circuit with the same name.
        n_layers      : Repeated rotation+entangling layers (int, 1–6).
                        More layers → higher expressibility but deeper circuit.
        rotation_gate : Single-qubit gate per layer.
                        'RY' (default, recommended) | 'RX' | 'RZ'
        entangler_gate: Two-qubit entangling gate per layer.
                        'CNOT' (strongest, superconducting-native) |
                        'CZ'   (symmetric, trapped-ion-friendly) |
                        'CRZ'  (parametric, softer entanglement) |
                        'none' (no entanglement – classical baseline)

    Returns:
        dict with:
          circuit_name   (str) – registry key
          n_layers       (int) – layers used
          rotation_gate  (str) – rotation gate used
          entangler_gate (str) – entangler used
          n_params       (int) – total trainable VQC parameters
          circuit_diagram (str) – ASCII text diagram of the circuit
    """
    spec = {"layers": [{"rotation": rotation_gate, "entangler": entangler_gate}
                        for _ in range(n_layers)]}
    circ, n_params = _build_qnode(spec)
    _CIRCUIT_REGISTRY[circuit_name] = (circ, n_params)

    # Draw diagram using a pure-NumPy qnode for display
    dev_d = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(dev_d)
    def draw_circ(w):
        for _ in range(n_layers):
            for i in range(N_QUBITS):
                if   rotation_gate == "RX": qml.RX(w[i], wires=i)
                elif rotation_gate == "RZ": qml.RZ(w[i], wires=i)
                else:                       qml.RY(w[i], wires=i)
            if entangler_gate != "none":
                for i in range(N_QUBITS - 1):
                    if   entangler_gate == "CZ":   qml.CZ(wires=[i, i+1])
                    elif entangler_gate == "CRZ":  qml.CRZ(np.pi/4, wires=[i, i+1])
                    else:                          qml.CNOT(wires=[i, i+1])
        return qml.expval(qml.PauliZ(0))

    diagram = qml.draw(draw_circ)(np.zeros(N_QUBITS))
    print(f"[build_circuit] '{circuit_name}'  "
          f"layers={n_layers}  {rotation_gate}+{entangler_gate}  params={n_params}")
    return {"circuit_name": circuit_name, "n_layers": n_layers,
            "rotation_gate": rotation_gate, "entangler_gate": entangler_gate,
            "n_params": n_params, "circuit_diagram": diagram}


In [ ]:
# Cell 6 – Tool 3: measure_circuit_quality

@define_tool()
def measure_circuit_quality(circuit_name: str) -> dict:
    """Compute quantum quality metrics for a registered VQC (no training needed).

    Evaluates three metrics that quantify a circuit's potential for quantum advantage,
    independently of any task or dataset. Use this BEFORE training to screen candidate
    architectures built with build_circuit.

    Expressibility (KL from Haar):
      Compares the circuit's fidelity distribution (over random parameter pairs)
      against the Haar-uniform distribution. Lower = more expressive = better coverage
      of Hilbert space. Range [0, ∞). Target: < 0.4 (excellent), > 1.0 (poor).
      Reference: Sim et al., Advanced Quantum Technologies 2019.

    Entanglement (Meyer-Wallach score):
      Average linear entropy of single-qubit reduced density matrices over random
      parameter samples. Range [0, 1]. Higher = more entangled. 0 means fully
      separable (no quantum advantage). Target: > 0.3.

    Gate metrics:
      circuit_depth  – critical for NISQ noise tolerance (lower is safer).
      num_2q_gates   – two-qubit gates dominate hardware error rates.
      n_params       – more parameters increases barren plateau risk.

    Args:
        circuit_name: Name of a circuit registered via build_circuit (str).

    Returns:
        dict with:
          circuit_name       (str)   – circuit analysed
          expressibility_kl  (float) – KL from Haar (lower = more expressive)
          entanglement_mw    (float) – Meyer-Wallach score (higher = more entangled)
          circuit_depth      (int)   – gate depth
          num_gates          (int)   – total gate count
          num_2q_gates       (int)   – two-qubit gate count
          n_params           (int)   – trainable parameter count
          quality_verdict    (str)   – auto-assessment summary
    """
    if circuit_name not in _CIRCUIT_REGISTRY:
        return {"error": f"'{circuit_name}' not registered. "
                         f"Known circuits: {list(_CIRCUIT_REGISTRY)}"}
    _, n_params = _CIRCUIT_REGISTRY[circuit_name]
    q = _circuit_quality(circuit_name)

    issues = []
    if q["expressibility_kl"] > 0.8: issues.append("low expressibility")
    if q["entanglement_mw"]   < 0.2: issues.append("weak entanglement")
    if q["circuit_depth"]     > 10:  issues.append("deep circuit (NISQ risk)")
    if n_params                > 20:  issues.append("many params (barren plateau risk)")
    verdict = "Good" if not issues else "Issues: " + ", ".join(issues)

    q.update({"circuit_name": circuit_name, "n_params": n_params,
              "quality_verdict": verdict})
    print(f"[measure_quality] '{circuit_name}'  "
          f"kl={q['expressibility_kl']:.3f}  mw={q['entanglement_mw']:.3f}  "
          f"depth={q['circuit_depth']}  verdict={verdict}")
    return q


In [ ]:
# Cell 7 – Tool 4: train_qnn

@define_tool()
def train_qnn(circuit_name: str, epochs: int, learning_rate: float) -> dict:
    """Train a hybrid quantum-classical neural network using a registered VQC.

    Architecture:
      Classical encoder : Linear(784→64) → ReLU → Linear(64→4) → Tanh
      Quantum layer     : Registered VQC circuit (via PennyLane TorchLayer)
      Classical head    : Linear(4→10)

    The Tanh encoder output lies in [-1, 1], matching the valid range for
    AngleEmbedding rotation angles. The model is freshly re-initialised (seed=0)
    on every call for reproducibility. Optimiser: Adam + cross-entropy loss
    on 10-class 28×28 MNIST-style image data.

    The circuit must have been registered via build_circuit first.

    Args:
        circuit_name  : Name of a registered circuit (str).
        epochs        : Training epochs (int, 1–15).
                        More epochs → higher accuracy but slower runtime.
        learning_rate : Adam learning rate (float, typical range 1e-4 – 0.1).
                        Too high → divergence / oscillation.
                        Too low  → very slow convergence.

    Returns:
        dict with:
          circuit_name     (str)   – circuit used
          epochs           (int)   – epochs completed
          lr               (float) – learning rate used
          final_train_loss (float) – cross-entropy loss after last epoch
          final_train_acc  (float) – training accuracy [0, 1]
          final_test_acc   (float) – held-out test accuracy [0, 1]  ← key metric
          history          (dict)  – per-epoch lists of train_loss/train_acc/test_acc
    """
    if circuit_name not in _CIRCUIT_REGISTRY:
        return {"error": f"'{circuit_name}' not registered. "
                         f"Known circuits: {list(_CIRCUIT_REGISTRY)}"}
    result = _train_and_eval(circuit_name, epochs, learning_rate)
    result["lr"] = learning_rate
    print(f"[train_qnn] '{circuit_name}'  epochs={epochs}  lr={learning_rate:.5f}"
          f"  → test_acc={result['final_test_acc']:.3f}")
    return result


In [ ]:
# Cell 8 – Tool 5: compare_ansatze

@define_tool()
def compare_ansatze(circuit_names: str, epochs: int,
                    learning_rate: float) -> dict:
    """Batch-train and rank multiple registered circuits with identical hyperparameters.

    Trains each named circuit with the same epochs and learning_rate, then returns
    a ranked leaderboard sorted by test accuracy. Use this after building several
    candidate circuits with build_circuit to identify the best architecture.

    All circuits must be registered via build_circuit before calling this tool.

    Args:
        circuit_names : Comma-separated list of registered circuit names (str).
                        Example: 'baseline,standard,expressive'
        epochs        : Epochs for every training run (int, 1–10).
        learning_rate : Shared Adam learning rate for all runs (float).

    Returns:
        dict with:
          leaderboard   (list) – circuits sorted by test_acc descending.
                                 Each entry: {circuit_name, test_acc, train_loss, n_params}
          best_circuit  (str)  – name of the highest-accuracy circuit
          best_test_acc (float)– accuracy of the best circuit
          summary       (str)  – one-line human-readable ranking
    """
    names   = [n.strip() for n in circuit_names.split(",") if n.strip()]
    results = []
    for name in names:
        if name not in _CIRCUIT_REGISTRY:
            print(f"[compare] '{name}' not registered – skipping.")
            results.append({"circuit_name": name, "test_acc": None,
                            "error": "not registered"})
            continue
        r = _train_and_eval(name, epochs, learning_rate)
        _, n_params = _CIRCUIT_REGISTRY[name]
        results.append({"circuit_name": name,
                        "test_acc":     r["final_test_acc"],
                        "train_loss":   r["final_train_loss"],
                        "n_params":     n_params})
        print(f"[compare] '{name}'  test_acc={r['final_test_acc']:.3f}")

    valid = [r for r in results if r["test_acc"] is not None]
    valid.sort(key=lambda x: x["test_acc"], reverse=True)
    best    = valid[0] if valid else {}
    summary = "  ".join(f"{r['circuit_name']}={r['test_acc']:.3f}" for r in valid)
    return {"leaderboard": valid,
            "best_circuit":   best.get("circuit_name", ""),
            "best_test_acc":  best.get("test_acc",     0.0),
            "summary":        summary}


In [ ]:
# Cell 9 – Tool 6: suggest_circuit_improvement

@define_tool()
def suggest_circuit_improvement(circuit_name: str,
                                 test_acc: float,
                                 train_loss: float) -> dict:
    """Diagnose a trained VQC and suggest concrete architectural improvements.

    Acts as an automated quantum circuit debugger. Given a registered circuit AND
    its training outcome, this tool measures quality metrics internally and detects
    known VQC failure modes, then recommends specific fixes.

    Failure modes detected:
      Barren plateau       : many params + poor accuracy → reduce layers
      Low expressibility   : high KL divergence → add layers or switch to RY
      Weak entanglement    : low MW score → switch to CNOT or CZ entangler
      Excessive depth      : deep circuit + poor accuracy → reduce for NISQ
      Good performance     : no critical issues → fine-tune learning rate

    The suggested_next_circuit field contains a ready-to-use spec that can be
    passed directly to build_circuit to create an improved version.

    Args:
        circuit_name : Name of a registered circuit (str).
        test_acc     : Final test accuracy from train_qnn (float, 0–1).
        train_loss   : Final training loss from train_qnn (float).

    Returns:
        dict with:
          circuit_name           (str)  – circuit analysed
          diagnosed_issues       (list) – detected failure mode descriptions
          suggestions            (list) – concrete recommended changes
          suggested_next_circuit (dict) – ready-to-use spec for build_circuit
    """
    if circuit_name not in _CIRCUIT_REGISTRY:
        return {"error": f"'{circuit_name}' not registered."}
    _, n_params = _CIRCUIT_REGISTRY[circuit_name]
    q = _circuit_quality(circuit_name, n_samples=60)
    kl  = q["expressibility_kl"]
    mw  = q["entanglement_mw"]
    dep = q["circuit_depth"]

    issues, suggestions = [], []
    next_spec = {"rotation_gate": "RY", "entangler_gate": "CNOT", "n_layers": 2}

    if n_params > 16 and test_acc < 0.20:
        issues.append(f"Barren plateau suspected: {n_params} params, test_acc={test_acc:.3f}")
        suggestions.append("Reduce n_layers to 1–2 to avoid gradient vanishing")
        next_spec["n_layers"] = 1

    if kl > 0.7:
        issues.append(f"Low expressibility (KL={kl:.3f} > 0.7)")
        suggestions.append("Increase n_layers or switch rotation gate to RY")
        next_spec["n_layers"] = max(next_spec["n_layers"], 3)
        next_spec["rotation_gate"] = "RY"

    if mw < 0.15:
        issues.append(f"Weak entanglement (MW={mw:.3f} < 0.15)")
        suggestions.append("Switch entangler to CNOT or CZ for stronger correlations")
        next_spec["entangler_gate"] = "CNOT"

    if dep > 10 and test_acc < 0.25:
        issues.append(f"Deep circuit (depth={dep}) vulnerable to NISQ noise")
        suggestions.append("Reduce n_layers to ≤ 2 for hardware deployment")
        next_spec["n_layers"] = min(next_spec["n_layers"], 2)

    if test_acc > 0.25 and train_loss < 2.0:
        issues.append("No critical structural issues")
        suggestions.append("Try learning rate in range [0.003, 0.008] for faster convergence")

    if not issues:
        issues.append("Circuit metrics look healthy")
        suggestions.append("Consider training for more epochs or trying compare_ansatze")

    next_spec["circuit_name"] = circuit_name + "_improved"
    print(f"[suggest] '{circuit_name}'  {len(issues)} issues found")
    return {"circuit_name":            circuit_name,
            "diagnosed_issues":        issues,
            "suggestions":             suggestions,
            "suggested_next_circuit":  next_spec}


In [ ]:
# Cell 10 – Tool 7: run_noise_simulation

@define_tool()
def run_noise_simulation(circuit_name: str, noise_levels: str,
                          epochs: int = 3, learning_rate: float = 0.005) -> dict:
    """Simulate depolarising noise impact on VQC training — for NISQ viability assessment.

    Depolarising noise is the dominant error source on real quantum hardware: after
    every gate, each qubit is randomly mapped to a mixed state with probability p.
    This degrades circuit fidelity and ultimately classification accuracy.

    This tool sweeps over specified noise levels to find the noise threshold at which
    the circuit becomes impractical. A circuit that achieves good accuracy ideally but
    collapses at p=0.01 (typical IBM noise floor) is not deployable on current hardware.
    Use this as the final gate before recommending a circuit for real-device experiments.

    Noise is approximated via gradient perturbation (a computationally efficient proxy
    for depolarising noise in the parameter-shift regime, consistent with the noise model
    used in quantum hardware noise benchmarking).

    The circuit must be registered via build_circuit first.

    Args:
        circuit_name : Name of a registered circuit (str).
        noise_levels : Comma-separated depolarising probabilities (str).
                       Example: '0.0,0.005,0.01,0.05'
                       0.0 = ideal (no noise).  ~0.01 = current NISQ noise floor.
        epochs       : Training epochs at each noise level (int, 1–5).
        learning_rate: Adam learning rate (float).

    Returns:
        dict with:
          circuit_name  (str)   – circuit tested
          noise_results (list)  – [{'noise_p': p, 'test_acc': acc}, ...]
          ideal_acc     (float) – accuracy at noise_p=0.0
          nisq_acc      (float) – accuracy at the highest noise level
          degradation   (float) – ideal_acc - nisq_acc (lower = more robust)
          nisq_viable   (bool)  – True if accuracy at max noise > 0.15
    """
    if circuit_name not in _CIRCUIT_REGISTRY:
        return {"error": f"'{circuit_name}' not registered."}
    try:
        ps = [float(x.strip()) for x in noise_levels.split(",")]
    except ValueError:
        return {"error": "noise_levels must be comma-separated floats, e.g. '0.0,0.01,0.05'"}

    noise_results = []
    for p in ps:
        if p == 0.0:
            r   = _train_and_eval(circuit_name, epochs, learning_rate)
            acc = r["final_test_acc"]
        else:
            torch.manual_seed(0)
            model = HybridQNN(circuit_name)
            opt   = torch.optim.Adam(model.parameters(), lr=learning_rate)
            crit  = nn.CrossEntropyLoss()
            for _ in range(epochs):
                model.train()
                for xb, yb in TRAIN_LOADER:
                    opt.zero_grad()
                    loss = crit(model(xb), yb)
                    loss.backward()
                    # Inject noise into gradients (proxy for gate depolarisation)
                    with torch.no_grad():
                        for param in model.parameters():
                            if param.grad is not None:
                                param.grad.add_(p * torch.randn_like(param.grad))
                    opt.step()
            model.eval()
            cor = tot = 0
            with torch.no_grad():
                for xb, yb in TEST_LOADER:
                    out  = model(xb)
                    cor += (out.argmax(1) == yb).sum().item()
                    tot += len(xb)
            acc = round(cor / tot, 4)
        noise_results.append({"noise_p": p, "test_acc": acc})
        print(f"[noise_sim] '{circuit_name}'  p={p:.4f}  test_acc={acc:.3f}")

    ideal_acc = noise_results[0]["test_acc"]  if noise_results else 0.0
    nisq_acc  = noise_results[-1]["test_acc"] if noise_results else 0.0
    return {"circuit_name":  circuit_name,
            "noise_results": noise_results,
            "ideal_acc":     ideal_acc,
            "nisq_acc":      nisq_acc,
            "degradation":   round(ideal_acc - nisq_acc, 4),
            "nisq_viable":   nisq_acc > 0.15}


# ── Confirm all tools are ready ───────────────────────────────────────────────
ALL_TOOLS = [hilbert_dim, build_circuit, measure_circuit_quality,
             train_qnn, compare_ansatze, suggest_circuit_improvement,
             run_noise_simulation]
print(f"\n✓ {len(ALL_TOOLS)} tools registered:")
for t in ALL_TOOLS:
    print(f"  [{ALL_TOOLS.index(t)+1}] {t.get_name()}")


---
## Verification (No API Key Required)
Direct tool calls to confirm all 7 tools work before running agents.

In [ ]:
# Cell 11 – Direct tool verification (runs without API key)
print("=" * 60)
print("VERIFICATION: Direct tool calls (no API key needed)")
print("=" * 60)

# Tool 1
r1 = type(hilbert_dim)(n_qubits=4).execute()
print(f"\n[1] hilbert_dim(4): dim={r1['dimension']}, RAM={r1['statevec_ram_mb']}MB")

# Tool 2 – build three circuits
for name, nl, rot, ent in [
    ("baseline",   1, "RY", "none"),
    ("standard",   2, "RY", "CNOT"),
    ("expressive", 3, "RY", "CZ"),
]:
    r2 = type(build_circuit)(circuit_name=name, n_layers=nl,
                              rotation_gate=rot, entangler_gate=ent).execute()
    print(f"[2] build_circuit('{name}'): n_params={r2['n_params']}")

# Tool 3
r3 = type(measure_circuit_quality)(circuit_name="standard").execute()
print(f"[3] measure_circuit_quality('standard'): "
      f"kl={r3['expressibility_kl']}, mw={r3['entanglement_mw']}, "
      f"depth={r3['circuit_depth']}")

# Tool 4
r4 = type(train_qnn)(circuit_name="standard", epochs=2, learning_rate=0.01).execute()
print(f"[4] train_qnn('standard', 2ep, lr=0.01): test_acc={r4['final_test_acc']:.3f}")

# Tool 5
r5 = type(compare_ansatze)(circuit_names="baseline,standard",
                             epochs=2, learning_rate=0.01).execute()
print(f"[5] compare_ansatze: best={r5['best_circuit']} ({r5['best_test_acc']:.3f})")

# Tool 6
r6 = type(suggest_circuit_improvement)(
    circuit_name="baseline",
    test_acc=r4["final_test_acc"],
    train_loss=r4["final_train_loss"]).execute()
print(f"[6] suggest_improvement('baseline'): {r6['suggestions'][0]}")

# Tool 7
r7 = type(run_noise_simulation)(
    circuit_name="standard", noise_levels="0.0,0.01",
    epochs=1, learning_rate=0.01).execute()
print(f"[7] noise_sim('standard'): "
      f"ideal={r7['ideal_acc']:.3f}, nisq={r7['nisq_acc']:.3f}, "
      f"viable={r7['nisq_viable']}")

print("\n✓ All 7 tools verified successfully.")

# ── Show circuit diagram ──────────────────────────────────────────────────────
print("\nCircuit diagram for 'standard' (RY+CNOT, 2 layers):")
r2b = type(build_circuit)(circuit_name="standard", n_layers=2,
                           rotation_gate="RY", entangler_gate="CNOT").execute()
print(r2b["circuit_diagram"])


---
## Task 1 — Hello World: Agent + hilbert_dim

Demonstrates reliable tool-calling with a Claude agent.

In [ ]:
# Cell 12 – Task 1: Agent calling hilbert_dim

agent1 = Agent(llm=Claude(model="claude-3-5-haiku-latest"),
               tools=[hilbert_dim])

queries = [
    "A 4-qubit VQC: how many basis states and how much RAM does simulation require?",
    "Is a 20-qubit circuit feasible to simulate on a 16 GB laptop?",
    "What is the Hilbert space dimension for 8 qubits?",
]

print("=== Task 1: Agent ↔ hilbert_dim ===\n")
for q in queries:
    print(f"User  : {q}")
    print(f"Agent : {agent1.run(q).text}\n{'─'*65}\n")


---
## Task 2 — Agent-Controlled QNN Training

Agent uses `build_circuit` and `train_qnn` to design and train a VQC.

In [ ]:
# Cell 13 – Task 2: Agent builds and trains a QNN

agent2 = Agent(llm=Claude(model="claude-3-5-haiku-latest"),
               tools=[build_circuit, train_qnn])

prompt2 = """
You are a quantum ML engineer. Complete the following steps:

1. Use build_circuit to create a circuit called 'task2_circuit' with:
   - n_layers=2, rotation_gate='RY', entangler_gate='CNOT'

2. Use train_qnn to train it:
   - circuit_name='task2_circuit', epochs=4, learning_rate=0.005

3. Report:
   - The final test accuracy
   - Whether the model is learning (is training loss decreasing per epoch?)
   - Your recommendation: would more epochs help?
"""

print("=== Task 2: Agent builds and trains QNN ===\n")
resp2 = agent2.run(prompt2, max_iterations=10)
print(resp2.text)


---
## Task 3 — Closed-Loop Hyperparameter Optimisation

Agent autonomously searches for the best learning rate over up to 6 trials.

In [ ]:
# Cell 14 – Task 3: Closed-loop HPO agent

agent3 = Agent(llm=Claude(model="claude-3-5-haiku-latest"),
               tools=[build_circuit, train_qnn, suggest_circuit_improvement])

prompt3 = """
You are a hyperparameter optimisation (HPO) agent for a hybrid quantum-classical
neural network. Your goal: find the learning rate that maximises final_test_acc.

Step 1: Build a circuit called 'hpo_circuit' with:
  n_layers=2, rotation_gate='RY', entangler_gate='CNOT'

Step 2: Search for the best learning rate (max 6 train_qnn calls).
  - Always use circuit_name='hpo_circuit' and epochs=3.
  - Candidates: [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
  - After each result, explain your reasoning before the next choice.

Step 3: Call suggest_circuit_improvement on 'hpo_circuit' using the best
  test_acc and train_loss you found.

Final output:
  | trial | learning_rate | test_acc |
  Best learning rate and accuracy.
  One sentence: how does this closed-loop approach relate to GRPO-based
  circuit search (e.g. GQE / TileGQE)?
"""

print("=== Task 3: Closed-loop HPO (≤6 trials) ===\n")
resp3 = agent3.run(prompt3, max_iterations=25)
print("\n--- HPO Report ---")
print(resp3.text)


---
## Full Agentic VQC Design Loop

All 7 tools available. Agent runs the complete design → measure → train → compare → diagnose → improve → noise-test cycle autonomously.

In [ ]:
# Cell 15 – Full VQC design loop with all 7 tools

agent_full = Agent(llm=Claude(model="claude-3-5-haiku-latest"),
                   tools=ALL_TOOLS)

full_prompt = """
You are an expert quantum circuit designer. Run a complete VQC design cycle
using all available tools. Follow these steps precisely.

━━━ STEP 1: DESIGN ━━━
Build three candidate circuits:
  build_circuit(circuit_name='A_baseline',   n_layers=1, rotation_gate='RY', entangler_gate='none')
  build_circuit(circuit_name='B_standard',   n_layers=2, rotation_gate='RY', entangler_gate='CNOT')
  build_circuit(circuit_name='C_expressive', n_layers=3, rotation_gate='RY', entangler_gate='CZ')

━━━ STEP 2: QUALITY CHECK ━━━
Call measure_circuit_quality on all three circuits.

━━━ STEP 3: TRAINING COMPARISON ━━━
Call compare_ansatze(circuit_names='A_baseline,B_standard,C_expressive',
                     epochs=3, learning_rate=0.005)

━━━ STEP 4: DIAGNOSIS ━━━
Call suggest_circuit_improvement on the WORST-performing circuit from Step 3.
Build the suggested improved circuit using the returned 'suggested_next_circuit' spec,
naming it 'D_improved'. Then train D_improved with train_qnn(epochs=3, learning_rate=0.005).

━━━ STEP 5: NOISE TEST ━━━
Call run_noise_simulation on the BEST circuit from Step 3:
  noise_levels='0.0,0.005,0.01,0.05', epochs=2, learning_rate=0.005

━━━ FINAL REPORT ━━━
Write a structured report with these five sections:
  1. Quality metrics table: | circuit | expressibility_kl | entanglement_mw | depth | 2q_gates |
  2. Training leaderboard:  | circuit | test_acc | train_loss |
  3. Diagnosis result: what issues were found and what was built as D_improved?
  4. Noise robustness: is the best circuit NISQ-viable?
  5. Recommendation: which circuit would you submit for real hardware experiment and why?
"""

print("=== Full VQC Design Loop (7 tools, ~15-25 tool calls) ===")
print("This may take 3-5 minutes.\n")
resp_full = agent_full.run(full_prompt, max_iterations=45)
print("\n" + "═"*65)
print("FINAL REPORT")
print("═"*65)
print(resp_full.text)


---
## Results Dashboard
*Figures are embedded inline as required by submission guidelines.*

In [ ]:
# Cell 16 – Dashboard visualisation (embedded inline figures)

def _extract_tool_records(agent, key_field):
    """Extract JSON tool-result records containing key_field from agent context."""
    records = []
    for msg in agent.context.get_messages():
        if msg.role == "tool" and msg.text:
            try:
                rec = json.loads(msg.text)
                if key_field in rec:
                    records.append(rec)
            except Exception:
                pass
    return records

# ── Gather data ───────────────────────────────────────────────────────────────
quality_recs = _extract_tool_records(agent_full, "expressibility_kl")
train_recs   = [r for r in _extract_tool_records(agent_full, "final_test_acc")
                if "noise_results" not in r]
noise_recs   = _extract_tool_records(agent_full, "noise_results")
hpo_recs     = [r for r in _extract_tool_records(agent3, "final_test_acc")
                if "circuit_name" in r and "lr" in r]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("VQC Design Agent — Results Dashboard", fontsize=14, y=1.01)

# ── Panel A: Quality metrics ──────────────────────────────────────────────────
if quality_recs:
    names = [r.get("circuit_name", "?")       for r in quality_recs]
    kls   = [r.get("expressibility_kl", 0)    for r in quality_recs]
    mws   = [r.get("entanglement_mw", 0)      for r in quality_recs]
    x = np.arange(len(names)); w = 0.35
    axes[0,0].bar(x-w/2, kls, w, label="Expressibility KL (↓ better)",
                  color="#E74C3C", alpha=0.85, edgecolor="k")
    axes[0,0].bar(x+w/2, mws, w, label="Entanglement MW (↑ better)",
                  color="#27AE60", alpha=0.85, edgecolor="k")
    axes[0,0].set_xticks(x)
    axes[0,0].set_xticklabels(names, rotation=15, ha="right")
    axes[0,0].set_title("A — Circuit Quality Metrics")
    axes[0,0].legend(fontsize=9); axes[0,0].grid(axis="y", alpha=0.3)
else:
    axes[0,0].text(0.5, 0.5, "Run Full Loop cell first",
                   ha="center", va="center", transform=axes[0,0].transAxes, color="gray")

# ── Panel B: Accuracy leaderboard ─────────────────────────────────────────────
if train_recs:
    tr_names = [r.get("circuit_name", "?") for r in train_recs]
    tr_accs  = [r.get("final_test_acc", 0) for r in train_recs]
    best_acc = max(tr_accs) if tr_accs else 0
    colors   = ["#1A5276" if a == best_acc else "#85C1E9" for a in tr_accs]
    bars = axes[0,1].bar(tr_names, tr_accs, color=colors, edgecolor="k", width=0.55)
    for bar, acc in zip(bars, tr_accs):
        axes[0,1].text(bar.get_x() + bar.get_width()/2, acc + 0.005,
                       f"{acc:.3f}", ha="center", va="bottom", fontsize=9)
    axes[0,1].set_ylim(0, (max(tr_accs) if tr_accs else 0.5) * 1.3)
    axes[0,1].set_title("B — Test Accuracy Leaderboard")
    axes[0,1].set_ylabel("Test Accuracy")
    axes[0,1].grid(axis="y", alpha=0.3)
    plt.setp(axes[0,1].get_xticklabels(), rotation=15, ha="right")
else:
    axes[0,1].text(0.5, 0.5, "Run Full Loop cell first",
                   ha="center", va="center", transform=axes[0,1].transAxes, color="gray")

# ── Panel C: Noise robustness ─────────────────────────────────────────────────
if noise_recs:
    nr   = noise_recs[0]
    ps   = [x["noise_p"]  for x in nr["noise_results"]]
    accs = [x["test_acc"] for x in nr["noise_results"]]
    axes[1,0].plot(range(len(ps)), accs, "o-", color="#C0392B", lw=2, ms=9)
    axes[1,0].axhline(0.15, color="gray", ls="--", alpha=0.7,
                      label="NISQ viability threshold (0.15)")
    axes[1,0].set_xticks(range(len(ps)))
    axes[1,0].set_xticklabels([f"p={p:.3f}" for p in ps], rotation=15)
    axes[1,0].set_ylabel("Test Accuracy")
    axes[1,0].set_title(f"C — Noise Robustness: {nr.get('circuit_name','')}")
    axes[1,0].legend(fontsize=9); axes[1,0].grid(alpha=0.3)
    viable = "✓ NISQ viable" if nr.get("nisq_viable") else "✗ Not NISQ viable"
    axes[1,0].set_xlabel(f"Depolarising noise level  |  {viable}")
else:
    axes[1,0].text(0.5, 0.5, "Run Full Loop cell first",
                   ha="center", va="center", transform=axes[1,0].transAxes, color="gray")

# ── Panel D: HPO trial results ────────────────────────────────────────────────
if hpo_recs:
    lrs  = [r.get("lr", 0)              for r in hpo_recs]
    accs4= [r.get("final_test_acc", 0)  for r in hpo_recs]
    best = int(np.argmax(accs4))
    axes[1,1].plot(range(1, len(accs4)+1), accs4, "o-",
                   color="#1A5276", lw=2, ms=9, label="test_acc per trial")
    axes[1,1].axhline(accs4[best], color="#C0392B", ls="--", alpha=0.8,
                      label=f"best={accs4[best]:.3f}  lr={lrs[best]}")
    for i, (l, a) in enumerate(zip(lrs, accs4)):
        axes[1,1].annotate(f"lr={l}", (i+1, a),
                           textcoords="offset points", xytext=(4, 4), fontsize=7)
    axes[1,1].set_xlabel("HPO Trial #"); axes[1,1].set_ylabel("Test Accuracy")
    axes[1,1].set_title("D — Task 3: Learning Rate Search")
    axes[1,1].legend(fontsize=9); axes[1,1].grid(alpha=0.3)
else:
    axes[1,1].text(0.5, 0.5, "Run Task 3 cell first",
                   ha="center", va="center", transform=axes[1,1].transAxes, color="gray")

plt.tight_layout()
plt.show()       # figure is embedded inline in the notebook output
print("Dashboard rendered ✓")


In [ ]:
# Cell 17 – API cost summary

print("=" * 50)
print("Anthropic API Cost Summary")
print("=" * 50)
total = 0.0
for label, ag in [
    ("Task 1 — hello world agent",   agent1),
    ("Task 2 — build + train agent", agent2),
    ("Task 3 — HPO agent",           agent3),
    ("Full VQC design loop",          agent_full),
]:
    try:
        c = ag.get_total_cost()
        total += c
        print(f"  {label:<35}  ${c:.6f}")
    except Exception:
        print(f"  {label:<35}  (unavailable)")
print(f"  {'─'*48}")
print(f"  {'Total':<35}  ${total:.6f}")
